# FastTextExtractor Test Notebook

This notebook demonstrates and tests the `FastTextExtractor` strategy for document extraction.

## What we'll test:
1. Document triage and classification
2. FastTextExtractor confidence scoring
3. Content extraction (text blocks, tables, figures)
4. Cost estimation
5. Sample extracted content display

## Setup and Imports

In [1]:
import sys
from pathlib import Path
from pprint import pprint

# Add src to path
sys.path.insert(0, str(Path().absolute().parent))

from src.agents.triage import TriageAgent
from src.strategies.fast_text import FastTextExtractor
from src.models.document_profile import DocumentProfile

c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Select Test Document

In [52]:
# Choose a test document
# Exact paths to the test documents
test_documents = {
    "class_a": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_a\EthSwitch-10th-Annual-Report-202324.pdf"),
    "class_b": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_b\2021_Audited_Financial_Statement_Report.pdf"),
    "class_c": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_c\fta_performance_survey_final_report_2022.pdf"),
    "class_d": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_d\tax_expenditure_ethiopia_2021_22.pdf"),
}

# Select which document to test
selected_doc = "class_d"  # Change this to test different documents: "class_a", "class_b", "class_c", or "class_d"
pdf_path = test_documents[selected_doc]

if not pdf_path.exists():
    print(f"⚠️  Document not found: {pdf_path}")
    print("Available documents:")
    for key, path in test_documents.items():
        exists = "✓" if path.exists() else "✗"
        print(f"  {exists} {key}: {path.name}")
else:
    print(f"✓ Selected document: {pdf_path.name}")
    print(f"  Path: {pdf_path}")
    print(f"  Size: {pdf_path.stat().st_size / 1024 / 1024:.2f} MB")

✓ Selected document: tax_expenditure_ethiopia_2021_22.pdf
  Path: C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_d\tax_expenditure_ethiopia_2021_22.pdf
  Size: 0.95 MB


## Step 1: Document Triage

In [53]:
# Initialize Triage Agent
profiles_dir = Path(".refinery/profiles")
profiles_dir.mkdir(parents=True, exist_ok=True)
triage = TriageAgent(profiles_dir=profiles_dir)

# Classify the document
print("🔍 Running Triage Agent...")
profile = triage.classify_document(pdf_path)

print("\n📋 Document Profile:")
print(f"  Document ID: {profile.doc_id}")
print(f"  Origin Type: {profile.origin_type}")
print(f"  Layout Complexity: {profile.layout_complexity}")
print(f"  Domain Hint: {profile.domain_hint}")
print(f"  Estimated Cost: {profile.estimated_cost}")
print(f"  Language: {profile.language} (confidence: {profile.language_confidence:.2f})")
print(f"  Page Count: {profile.metadata.page_count}")
print(f"  File Size: {profile.metadata.size_bytes / 1024 / 1024:.2f} MB")

🔍 Running Triage Agent...

📋 Document Profile:
  Document ID: tax_expenditure_ethiopia_2021_22
  Origin Type: native_digital
  Layout Complexity: table_heavy
  Domain Hint: financial
  Estimated Cost: needs_layout_model
  Language: en (confidence: 1.00)
  Page Count: 60
  File Size: 0.95 MB


## Step 2: Initialize FastTextExtractor

In [54]:
# Create extractor instance
extractor = FastTextExtractor()

print(f"✓ Initialized: {extractor.name} extractor")
print(f"\n🔍 Checking if extractor can handle this document...")
can_handle = extractor.can_handle(profile)
print(f"  Can Handle: {can_handle}")

if not can_handle:
    print("\n⚠️  Warning: FastTextExtractor may not be optimal for this document")
    print("   Consider using a layout-aware or vision-augmented strategy instead")
else:
    print("\n✓ This document is suitable for fast text extraction")

✓ Initialized: fast_text extractor

🔍 Checking if extractor can handle this document...
  Can Handle: False

⚠️  Warning: FastTextExtractor may not be optimal for this document
   Consider using a layout-aware or vision-augmented strategy instead


## Step 3: Confidence Scoring

In [55]:
print("📊 Calculating confidence score...")
confidence = extractor.confidence_score(str(pdf_path))

print(f"\n  Confidence: {confidence:.4f} ({confidence*100:.1f}%)")

# Interpret confidence
if confidence >= 0.8:
    interpretation = "Very High - Extraction should work excellently"
elif confidence >= 0.6:
    interpretation = "High - Extraction should work well"
elif confidence >= 0.4:
    interpretation = "Medium - Extraction may have some issues"
else:
    interpretation = "Low - Consider using a different strategy"

print(f"  Interpretation: {interpretation}")

📊 Calculating confidence score...

  Confidence: 0.6498 (65.0%)
  Interpretation: High - Extraction should work well


## Step 4: Cost Estimation

In [56]:
print("💰 Estimating extraction cost...")
cost = extractor.cost_estimate(str(pdf_path))

print(f"\n  Total Cost: ${cost['total_cost_usd']:.4f}")
print(f"  Cost per Page: ${cost['cost_per_page']:.4f}")
print(f"\n  💡 Fast text extraction is CPU-only and effectively free!")

💰 Estimating extraction cost...

  Total Cost: $0.0000
  Cost per Page: $0.0000

  💡 Fast text extraction is CPU-only and effectively free!


## Step 5: Extract Content

In [57]:
print("📄 Extracting content from document...")
print("   This may take a moment for large documents...")

import time
start_time = time.time()

extracted = extractor.extract(str(pdf_path))

elapsed = time.time() - start_time

print(f"\n✓ Extraction completed in {elapsed:.2f} seconds")
print(f"\n📊 Extraction Summary:")
print(f"  Text Blocks: {len(extracted.text_blocks):,}")
print(f"  Tables: {len(extracted.tables)}")
print(f"  Figures: {len(extracted.figures)}")
print(f"  Reading Order: {len(extracted.reading_order)} indices")

📄 Extracting content from document...
   This may take a moment for large documents...

✓ Extraction completed in 17.30 seconds

📊 Extraction Summary:
  Text Blocks: 16,348
  Tables: 43
  Figures: 2
  Reading Order: 16348 indices


## Step 6: Sample Text Blocks

In [58]:
print("📝 Sample Text Blocks (first 10):")
print("=" * 80)

for i, block in enumerate(extracted.text_blocks[:10], 1):
    content = block.content.strip()
    if len(content) > 60:
        content = content[:60] + "..."
    
    print(f"\n[{i}] Page {block.page_num}")
    print(f"    Content: {content!r}")
    print(f"    BBox: ({block.bbox.x0:.1f}, {block.bbox.y0:.1f}) → ({block.bbox.x1:.1f}, {block.bbox.y1:.1f})")
    print(f"    Size: {len(block.content)} chars")

📝 Sample Text Blocks (first 10):

[1] Page 1
    Content: 'Import'
    BBox: (87.6, 420.7) → (149.9, 440.7)
    Size: 6 chars

[2] Page 1
    Content: 'tax'
    BBox: (155.4, 420.7) → (184.4, 440.7)
    Size: 3 chars

[3] Page 1
    Content: 'expenditure'
    BBox: (190.0, 420.7) → (303.4, 440.7)
    Size: 11 chars

[4] Page 1
    Content: 'report'
    BBox: (308.9, 420.7) → (366.7, 440.7)
    Size: 6 chars

[5] Page 1
    Content: ':'
    BBox: (366.8, 423.9) → (372.1, 439.9)
    Size: 1 chars

[6] Page 1
    Content: 'FY'
    BBox: (87.6, 446.0) → (113.2, 466.0)
    Size: 2 chars

[7] Page 1
    Content: '2018/19'
    BBox: (118.7, 446.0) → (191.2, 466.0)
    Size: 7 chars

[8] Page 1
    Content: '–'
    BBox: (196.7, 446.0) → (207.9, 466.0)
    Size: 1 chars

[9] Page 1
    Content: '2020/21'
    BBox: (213.3, 446.0) → (285.8, 466.0)
    Size: 7 chars

[10] Page 1
    Content: 'Tax'
    BBox: (87.6, 584.4) → (104.9, 595.4)
    Size: 3 chars


## Step 7: Sample Tables

In [59]:
if extracted.tables:
    print(f"📊 Sample Tables (showing first {min(3, len(extracted.tables))}):")
    print("=" * 80)
    
    for i, table in enumerate(extracted.tables[:3], 1):
        print(f"\n[Table {i}] Page {table.page_num}")
        print(f"  Headers ({len(table.headers)}): {table.headers}")
        print(f"  Rows: {len(table.rows)}")
        
        if table.rows:
            print(f"  First Row: {table.rows[0]}")
            if len(table.rows) > 1:
                print(f"  Second Row: {table.rows[1]}")
        
        print(f"  BBox: ({table.bbox.x0:.1f}, {table.bbox.y0:.1f}) → ({table.bbox.x1:.1f}, {table.bbox.y1:.1f})")
else:
    print("📊 No tables detected in this document")

📊 Sample Tables (showing first 3):

[Table 1] Page 16
  Headers (5): ['None', 'None', 'None', 'None', '']
  Rows: 9
  First Row: ['', '', 'None', 'None', '']
  Second Row: ['', 'None', 'None', 'None', 'None']
  BBox: (170.3, 322.5) → (468.1, 454.9)

[Table 2] Page 17
  Headers (8): ['None', 'None', 'None', 'None', 'None', 'None', 'None', '']
  Rows: 8
  First Row: ['None', '', '', 'None', 'None', 'None', 'None', '']
  Second Row: ['None', 'None', 'None', 'None', '', 'None', 'None', 'None']
  BBox: (143.8, 256.6) → (477.1, 403.7)

[Table 3] Page 18
  Headers (10): ['', '', 'None', 'None', '', 'Imports (CIF value)', '', '', 'Total expenditure', '']
  Rows: 9
  First Row: ['2018/19', '', 'Capital/investment', '', '', '2,065.91', '', '', '711.02', '']
  Second Row: ['None', 'Second schedule', 'None', 'None', '23,929.54', 'None', 'None', '4,006.69', 'None', 'None']
  BBox: (82.5, 152.3) → (523.6, 366.9)


## Step 8: Sample Figures

In [60]:
if extracted.figures:
    print(f"🖼️  Sample Figures (showing first {min(5, len(extracted.figures))}):")
    print("=" * 80)
    
    for i, figure in enumerate(extracted.figures[:5], 1):
        print(f"\n[Figure {i}] Page {figure.page_num}")
        print(f"  Caption: {figure.caption or '(none)'}")
        print(f"  BBox: ({figure.bbox.x0:.1f}, {figure.bbox.y0:.1f}) → ({figure.bbox.x1:.1f}, {figure.bbox.y1:.1f})")
        width = figure.bbox.x1 - figure.bbox.x0
        height = figure.bbox.y1 - figure.bbox.y0
        print(f"  Size: {width:.1f} × {height:.1f} points")
else:
    print("🖼️  No figures/images detected in this document")

🖼️  Sample Figures (showing first 2):

[Figure 1] Page 1
  Caption: (none)
  BBox: (239.8, 108.3) → (354.8, 217.8)
  Size: 115.0 × 109.5 points

[Figure 2] Page 2
  Caption: (none)
  BBox: (408.5, 661.3) → (510.6, 749.4)
  Size: 102.1 × 88.1 points


## Step 9: Text Block Statistics

In [61]:
import statistics
from collections import Counter

if extracted.text_blocks:
    # Calculate statistics
    char_counts = [len(block.content) for block in extracted.text_blocks]
    pages = [block.page_num for block in extracted.text_blocks]
    
    print("📈 Text Block Statistics:")
    print("=" * 80)
    print(f"  Total Text Blocks: {len(extracted.text_blocks):,}")
    print(f"  Total Characters: {sum(char_counts):,}")
    print(f"  Average Chars per Block: {statistics.mean(char_counts):.1f}")
    print(f"  Median Chars per Block: {statistics.median(char_counts):.1f}")
    print(f"  Min Chars: {min(char_counts)}")
    print(f"  Max Chars: {max(char_counts)}")
    print(f"  Pages Covered: {min(pages)} to {max(pages)}")
    
    # Blocks per page
    blocks_per_page = Counter(pages)
    print(f"\n  Blocks per Page (sample):")
    for page_num in sorted(blocks_per_page.keys())[:10]:
        print(f"    Page {page_num}: {blocks_per_page[page_num]} blocks")

📈 Text Block Statistics:
  Total Text Blocks: 16,348
  Total Characters: 88,735
  Average Chars per Block: 5.4
  Median Chars per Block: 5.0
  Min Chars: 1
  Max Chars: 131
  Pages Covered: 1 to 60

  Blocks per Page (sample):
    Page 1: 18 blocks
    Page 2: 117 blocks
    Page 3: 149 blocks
    Page 4: 415 blocks
    Page 5: 253 blocks
    Page 6: 401 blocks
    Page 7: 308 blocks
    Page 8: 109 blocks
    Page 9: 413 blocks
    Page 10: 474 blocks


## Step 10: Full Document Text Preview

In [62]:
# Combine all text blocks into a single document text
full_text = " ".join(block.content for block in extracted.text_blocks)

print(f"📄 Full Document Text Preview:")
print("=" * 80)
print(f"\nTotal Length: {len(full_text):,} characters")
print(f"\nFirst 500 characters:")
print("-" * 80)
print(full_text[:500])
print("-" * 80)
print(f"\n... (truncated) ...")

📄 Full Document Text Preview:

Total Length: 105,082 characters

First 500 characters:
--------------------------------------------------------------------------------
Import tax expenditure report : FY 2018/19 – 2020/21 Tax Policy Directorate, Ministry of Finance, Ethiopia September 2022 Import tax expenditure report: FY 2018/19 – 2020/21  Ministry of Finance, Ethiopia 2 Preface This report has been prepared by the IFS Centre for Tax Analysis in Developing Countries (TaxDev) team (Daniel Prinz, Edris Seid and Ben Waltmann) in collaboration with the Ethiopian Ministry of Finance Tax Policy Directorate team (led by Mulay Weldu). The authors would like to thank
--------------------------------------------------------------------------------

... (truncated) ...


## Summary

In [63]:
print("✅ Test Summary:")
print("=" * 80)
print(f"\nDocument: {pdf_path.name}")

print("\nProfile:")
print(f"  - Document ID: {profile.doc_id}")
print(f"  - Origin: {profile.origin_type}  (native_digital = has a real text layer; scanned_image = image-only; mixed = both)")
print(f"  - Layout: {profile.layout_complexity}  (multi_column = multiple text columns detected via x-position clustering)")
print(f"  - Domain: {profile.domain_hint}  (financial = keyword hits on revenue/profit/balance sheet/etc.)")
print(f"  - Estimated Cost: {profile.estimated_cost}  (needs_layout_model = multi-column / table-heavy / figure-heavy layouts)")
print(
    f"  - Language: {profile.language} (confidence: {profile.language_confidence:.2f})  "
    "(heuristic over Unicode ranges; detects Ethiopic/Amharic vs default English)"
)
print(f"  - Page Count: {profile.metadata.page_count}  (pages = len(pdf.pages))")
print(
    f"  - File Size: {profile.metadata.size_bytes / 1024 / 1024:.2f} MB  "
    "(bytes from filesystem; useful for cost/performance expectations)"
)

print("\nExtraction:")
print(f"  - Strategy: {extractor.name}  (uses pdfplumber as the primary parser for text/tables/images)")
print(f"  - Can Handle: {can_handle}  (True only for native_digital + single_column profiles)")
print(f"  - Confidence: {confidence:.2%}  (combined character-density/layout/table scores)")
print(f"  - Cost: ${cost['total_cost_usd']:.4f}  (fast text is CPU-only and effectively free)")

print("\nResults:")
print(f"  - Text Blocks: {len(extracted.text_blocks):,}")
print(f"  - Tables: {len(extracted.tables)}")
print(f"  - Figures: {len(extracted.figures)}")

print("\nDetails:")
print(
    "  - Origin type heuristics: avg_chars/page, image-to-page area ratio, "
    "and font coverage (chars layer). "
    "scanned_image if chars < 50/page AND images > 80% of page area; "
    "mixed if images are high or font coverage is low; otherwise native_digital."
)
print(
    "  - Layout complexity: estimate columns by clustering x-positions into "
    "4 bins; count table-like rows and image pages to choose between "
    "single_column, multi_column, table_heavy, and figure_heavy."
)
print(
    "  - Domain hint: keyword hits over the first ~10 pages (financial/legal/" 
    "technical/medical); best-scoring domain wins with a simple confidence "
    "scaling."
)
print(
    "  - Language: sample text from first pages; measure proportion of Ethiopic "
    "(Amharic) Unicode codepoints vs others to choose 'am' vs 'en'."
)
print(
    "  - Estimated cost: scanned_image → needs_vision_model; complex layouts "
    "(multi_column/table_heavy/figure_heavy/mixed) → needs_layout_model; "
    "simple single_column native_digital → fast_text_sufficient."
)

print("\n✓ FastTextExtractor test completed successfully!")

✅ Test Summary:

Document: tax_expenditure_ethiopia_2021_22.pdf

Profile:
  - Document ID: tax_expenditure_ethiopia_2021_22
  - Origin: native_digital  (native_digital = has a real text layer; scanned_image = image-only; mixed = both)
  - Layout: table_heavy  (multi_column = multiple text columns detected via x-position clustering)
  - Domain: financial  (financial = keyword hits on revenue/profit/balance sheet/etc.)
  - Estimated Cost: needs_layout_model  (needs_layout_model = multi-column / table-heavy / figure-heavy layouts)
  - Language: en (confidence: 1.00)  (heuristic over Unicode ranges; detects Ethiopic/Amharic vs default English)
  - Page Count: 60  (pages = len(pdf.pages))
  - File Size: 0.95 MB  (bytes from filesystem; useful for cost/performance expectations)

Extraction:
  - Strategy: fast_text  (uses pdfplumber as the primary parser for text/tables/images)
  - Can Handle: False  (True only for native_digital + single_column profiles)
  - Confidence: 64.98%  (combined cha